# Experiment 14: RNN Model for Stock Price Prediction

**Language:** Python  
**Duration:** 1 hour

This notebook implements a simple **RNN/LSTM model** for stock price prediction.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN, Dropout

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load Dataset

In [ ]:
# Use your own CSV file if available
# Example:
# df = pd.read_csv("stock_prices.csv")

# Demo dataset
dates = pd.date_range(start="2020-01-01", periods=500, freq="D")
trend = np.linspace(100, 180, 500)
seasonal = 5 * np.sin(np.linspace(0, 20, 500))
noise = np.random.normal(0, 1.5, 500)

df = pd.DataFrame({
    "Date": dates,
    "Close": trend + seasonal + noise
})

df.head()

## 3. Preprocess Data

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

data = df[["Close"]].values
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

scaled_data[:5]

## 4. Create Sequences

In [ ]:
def create_sequences(dataset, time_step=60):
    X, y = [], []
    for i in range(time_step, len(dataset)):
        X.append(dataset[i-time_step:i, 0])
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

time_step = 60
X, y = create_sequences(scaled_data, time_step)
X = X.reshape((X.shape[0], X.shape[1], 1))

print("X shape:", X.shape)
print("y shape:", y.shape)

## 5. Split Train and Test

In [ ]:
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 6. Build LSTM Model

In [ ]:
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

## 7. Train Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

## 8. Predict

In [ ]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_pred = scaler.inverse_transform(train_pred)
test_pred = scaler.inverse_transform(test_pred)

y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

## 9. Evaluate

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_actual, test_pred))
mae = mean_absolute_error(y_test_actual, test_pred)

print("RMSE:", rmse)
print("MAE:", mae)

## 10. Plot Results

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test_actual, label="Actual Price")
plt.plot(test_pred, label="Predicted Price")
plt.title("Actual vs Predicted Stock Price")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

## 11. Optional Simple RNN Model

In [ ]:
simple_rnn_model = Sequential([
    SimpleRNN(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    SimpleRNN(50),
    Dropout(0.2),
    Dense(1)
])

simple_rnn_model.compile(optimizer="adam", loss="mean_squared_error")
simple_rnn_model.summary()

## 12. Conclusion

- This notebook predicts stock prices using past closing prices.
- It uses sequence data as input to an LSTM/RNN model.
- LSTM usually performs better than a plain RNN for time-series tasks.